# Predict NFL passing yards from public data with Nori

Run this notebook from a fresh Jupyter/Colab environment. It downloads football data and the public model, builds Q1/halftime features, generates fresh predictions, and measures forecast error. No saved predictions, internal files, or private checkpoint are required.

This portable model is a new baseline with a smaller feature set and explicit context cap. It does **not** claim to reproduce the blog’s selected 24.1% strategy. The default runs one test week; change MAX_WEEK to 18 for the complete season. Kalshi replay is optional and disabled by default.

In [ ]:
import sys, subprocess, importlib.metadata as metadata
required = {'synthefy-nori':'0.19.0', 'nflreadpy':'0.1.5'}
for package, version in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package+'=='+version])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'pyarrow', 'requests', 'matplotlib'])
print({p:metadata.version(p) for p in required})

## Download and feature-building implementation
The implementation is included below so a downloaded notebook works without fetching helper scripts. nflreadpy retrieves nflverse play-by-play and official weekly player statistics. Each source is cached with a content hash.

Rows use the first observed QB passer for each team by the checkpoint. Features include current passing usage and score, prior QB checkpoint production over three/eight games, and prior offense/defense summaries. QB, team and opponent are encoded using only model-context rows. Current-week outcomes are excluded from historical features. Injury/replacement outcomes remain in evaluation. Historical corrected play timestamps do not establish when a live data feed published each statistic.

In [ ]:
import types
import sys
pipeline = types.ModuleType("nfl_passing_yards_pipeline")
sys.modules[pipeline.__name__] = pipeline
exec(compile("\"\"\"Public-source Q1/halftime NFL passing-yard modeling baseline.\n\nDownloads nflverse via nflreadpy; no saved predictions or private artifacts.\nRetrospectively corrected play data and play timestamps are NOT a real-time\npublication feed. This new baseline does not reproduce the blog's selected model.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport nflreadpy as nfl\nimport numpy as np\nimport pandas as pd\nfrom sklearn.preprocessing import OrdinalEncoder\nfrom synthefy_nori import NoriRegressor\n\n\nCATEGORIES = [\"actual_qb_id\", \"team\", \"opponent_team\"]\nLIVE = [\"yards_so_far\", \"attempts_so_far\", \"dropbacks_so_far\", \"sacks_so_far\",\n        \"epa_per_dropback\", \"cpoe\", \"air_yards_per_attempt\", \"sack_rate\", \"ypa\",\n        \"offense_plays\", \"offense_pass_rate\", \"offense_epa\", \"score_margin\"]\n\n\ndef _load(cache_dir, kind, season):\n    path = Path(cache_dir) / f\"{kind}_{season}.parquet\"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not path.exists():\n        print(f\"Downloading nflverse {kind} {season}\", flush=True)\n        loader = nfl.load_pbp if kind == \"pbp\" else nfl.load_player_stats\n        loader([season]).write_parquet(path)\n    manifest_path = Path(cache_dir) / f\"{kind}_{season}.source.json\"\n    manifest_path.write_text(json.dumps({\"loader\": f\"nflreadpy.load_{kind}\", \"season\": season,\n        \"sha256\": hashlib.sha256(path.read_bytes()).hexdigest(), \"bytes\": path.stat().st_size}, indent=2))\n    return pd.read_parquet(path)\n\n\ndef _sum(frame, col):\n    return float(frame[col].fillna(0).sum())\n\n\ndef _mean(frame, col):\n    return float(frame[col].mean()) if len(frame) else np.nan\n\n\ndef build_dataset(cache_dir, seasons, decision_delay_seconds=120):\n    \"\"\"One first-passer/team/game/checkpoint row, with earlier-week history.\n\n    QB selection uses the first identified passer by the checkpoint, never final\n    attempts. Injury/replacement outcomes remain in evaluation. Missing official\n    labels are excluded, not fabricated. No final-game weather or betting lines\n    are features because their historical publication times are unknown.\n    \"\"\"\n    if decision_delay_seconds < 0:\n        raise ValueError(\"decision_delay_seconds must be nonnegative\")\n    records = []\n    for season in sorted(set(seasons)):\n        pbp = _load(cache_dir, \"pbp\", season)\n        stats = _load(cache_dir, \"player_stats\", season)\n        stats = stats.loc[stats.season_type == \"REG\"]\n        qb_ids = set(stats.loc[stats.position == \"QB\", \"player_id\"])\n        stats = stats.set_index([\"game_id\", \"player_id\"])\n        pbp = pbp.loc[pbp.season_type == \"REG\"].copy()\n        pbp[\"_utc\"] = pd.to_datetime(pbp.time_of_day, utc=True, errors=\"coerce\")\n        for game_id, game in pbp.groupby(\"game_id\", sort=True):\n            game = game.sort_values(\"play_id\")\n            for quarter, checkpoint in [(1, \"q1\"), (2, \"halftime\")]:\n                quarter_rows = game.loc[game.qtr == quarter]\n                anchor = quarter_rows._utc.max()\n                if pd.isna(anchor):\n                    continue\n                seen = game.loc[game.qtr.between(1, quarter) & (game._utc <= anchor)]\n                if seen.empty:\n                    continue\n                last = seen.iloc[-1]\n                for team in [last.home_team, last.away_team]:\n                    offense = seen.loc[seen.posteam == team]\n                    passers = offense.loc[offense.passer_player_id.isin(qb_ids)]\n                    if passers.empty:\n                        continue\n                    qb_id = passers.iloc[0].passer_player_id\n                    if (game_id, qb_id) not in stats.index:\n                        continue\n                    label = stats.loc[(game_id, qb_id)]\n                    if isinstance(label, pd.DataFrame):\n                        raise ValueError(f\"Duplicate official label: {game_id}/{qb_id}\")\n                    qb = offense.loc[(offense.passer_player_id == qb_id) |\n                                     ((offense.rusher_player_id == qb_id) & (offense.qb_dropback == 1))]\n                    drops = qb.loc[qb.qb_dropback == 1]\n                    attempts = qb.loc[(qb.pass_attempt == 1) & (qb.sack != 1) & (qb.two_point_attempt != 1)]\n                    valid = offense.loc[offense.play_type.isin([\"pass\", \"run\"])]\n                    yards = _sum(qb.loc[qb.two_point_attempt != 1], \"passing_yards\")\n                    home = team == last.home_team\n                    rec = dict(game_id=game_id, game_date=str(last.game_date), season=int(season),\n                               week=int(last.week), team=team,\n                               opponent_team=last.away_team if home else last.home_team,\n                               actual_qb_id=qb_id, actual_qb_name=label.player_display_name,\n                               checkpoint=checkpoint, live_anchor_utc=anchor,\n                               game_end_utc=game._utc.max() + pd.Timedelta(minutes=5),\n                               decision_utc=anchor + pd.Timedelta(seconds=decision_delay_seconds),\n                               official_passing_yards=float(label.passing_yards),\n                               home=int(home), seconds_remaining=3600-quarter*900,\n                               yards_so_far=yards, attempts_so_far=float(len(attempts)),\n                               dropbacks_so_far=float(len(drops)), sacks_so_far=_sum(qb, \"sack\"),\n                               epa_per_dropback=_mean(drops, \"epa\"), cpoe=_mean(attempts, \"cpoe\"),\n                               air_yards_per_attempt=_mean(attempts, \"air_yards\"),\n                               sack_rate=_sum(qb, \"sack\") / max(len(drops), 1),\n                               ypa=yards / max(len(attempts), 1), offense_plays=float(len(valid)),\n                               offense_pass_rate=_mean(valid, \"qb_dropback\"), offense_epa=_mean(valid, \"epa\"),\n                               score_margin=float(last.total_home_score-last.total_away_score)*(1 if home else -1))\n                    rec[\"remaining_yards\"] = rec[\"official_passing_yards\"] - yards\n                    records.append(rec)\n    rows = pd.DataFrame(records).sort_values([\"season\", \"week\", \"game_id\", \"checkpoint\"]).reset_index(drop=True)\n    if rows.empty:\n        raise ValueError(\"No timestamped checkpoint rows with official labels\")\n    return add_history(rows)\n\n\ndef add_history(rows):\n    \"\"\"Group matching-checkpoint histories; embargo the entire current week.\"\"\"\n    rows = rows.copy()\n    for checkpoint, idx in rows.groupby(\"checkpoint\").groups.items():\n        part = rows.loc[idx]\n        for index, row in part.iterrows():\n            prior = part.loc[((part.season < row.season) | ((part.season == row.season) & (part.week < row.week)))\n                             & (part.game_end_utc < row.live_anchor_utc)]\n            qb = prior.loc[prior.actual_qb_id == row.actual_qb_id]\n            for window in [3, 8]:\n                hist = qb.tail(window)\n                for col in LIVE + [\"remaining_yards\", \"official_passing_yards\"]:\n                    rows.loc[index, f\"qb_prior{window}_{col}\"] = hist[col].mean()\n            rows.loc[index, \"qb_prior_games\"] = len(qb)\n            if not qb.empty:\n                rows.loc[index, \"rest_days\"] = (pd.Timestamp(row.game_date) - pd.Timestamp(qb.iloc[-1].game_date)).days\n            for name, key, value in [(\"offense\", \"team\", row.team), (\"defense\", \"opponent_team\", row.opponent_team)]:\n                hist = prior.loc[prior[key] == value].tail(8)\n                for col in [\"ypa\", \"epa_per_dropback\", \"sack_rate\", \"offense_pass_rate\", \"offense_plays\"]:\n                    rows.loc[index, f\"{name}_prior8_{col}\"] = hist[col].mean()\n    return rows\n\n\ndef feature_columns(rows):\n    return [\"week\", \"home\", \"seconds_remaining\"] + LIVE + [c for c in rows if \"_prior\" in c or c == \"rest_days\"]\n\n\ndef predict_weekly(rows, test_season, max_week=18, device=\"cpu\", context_limit=512,\n                   min_week=1, model=\"nori-6m\", weekly_update=True):\n    \"\"\"Fresh public Nori predictions, independently per checkpoint and week.\"\"\"\n    if context_limit < 2:\n        raise ValueError(\"context_limit must be >= 2\")\n    outputs = []\n    features = feature_columns(rows)\n    model_instance = NoriRegressor(model=model, device=device)\n    for week in range(min_week, max_week + 1):\n        for checkpoint in [\"q1\", \"halftime\"]:\n            available = rows.loc[rows.checkpoint == checkpoint]\n            train = available.loc[(available.season < test_season) |\n                                  (weekly_update & (available.season == test_season) & (available.week < week))]\n            train = train.sort_values([\"season\", \"week\", \"game_id\"]).tail(context_limit)\n            test = available.loc[(available.season == test_season) & (available.week == week)].copy()\n            if test.empty:\n                continue\n            train = train.loc[train.game_end_utc < test.live_anchor_utc.min()]\n            if len(train) < 2:\n                raise ValueError(f\"Insufficient prior context for {test_season} week {week}\")\n            encoder = OrdinalEncoder(handle_unknown=\"use_encoded_value\", unknown_value=-1)\n            xc = encoder.fit_transform(train[CATEGORIES].fillna(\"unknown\"))\n            xq = encoder.transform(test[CATEGORIES].fillna(\"unknown\"))\n            xtrain = np.column_stack([train[features].to_numpy(dtype=float), xc])\n            xtest = np.column_stack([test[features].to_numpy(dtype=float), xq])\n            print(f\"Nori {test_season} W{week} {checkpoint}: {len(train)} context, {len(test)} queries\", flush=True)\n            model_instance.fit(xtrain, train.remaining_yards.to_numpy())\n            dist = model_instance.predict(xtest, output_type=\"full\")\n            quantiles = np.asarray(dist[\"quantiles\"]) + test.yards_so_far.to_numpy()[:, None]\n            taus = np.asarray(dist[\"taus\"])\n            test[\"predicted_mean\"] = np.asarray(dist[\"mean\"]).reshape(-1) + test.yards_so_far.to_numpy()\n            test[\"quantile_levels\"] = [taus.tolist() for _ in range(len(test))]\n            test[\"quantile_values\"] = quantiles.tolist()\n            test[\"context_rows\"] = len(train)\n            test[\"model\"] = model\n            for tau, label in [(0.1, \"p10\"), (0.5, \"p50\"), (0.9, \"p90\")]:\n                test[label] = [np.interp(tau, taus, q) for q in quantiles]\n            outputs.append(test)\n    if not outputs:\n        raise ValueError(\"No query rows in requested season/week range\")\n    return pd.concat(outputs, ignore_index=True)\n\n\ndef prediction_metrics(predictions):\n    results = []\n    for checkpoint, part in predictions.groupby(\"checkpoint\"):\n        actual = part.official_passing_yards.to_numpy()\n        error = part.predicted_mean.to_numpy() - actual\n        q = np.stack(part.quantile_values)\n        taus = np.asarray(part.iloc[0].quantile_levels)\n        residual = actual[:, None] - q\n        results.append(dict(checkpoint=checkpoint, rows=len(part), mae=float(np.abs(error).mean()),\n                            rmse=float(np.sqrt((error**2).mean())),\n                            pinball_loss=float(np.maximum(taus*residual, (taus-1)*residual).mean()),\n                            p10_p90_coverage=float(((actual >= part.p10) & (actual <= part.p90)).mean())))\n    return pd.DataFrame(results)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--cache-dir\", default=\"data/nfl_passing_yards\")\n    parser.add_argument(\"--start-season\", type=int, default=2023)\n    parser.add_argument(\"--test-season\", type=int, default=2025)\n    parser.add_argument(\"--max-week\", type=int, default=1)\n    parser.add_argument(\"--context-limit\", type=int, default=128)\n    parser.add_argument(\"--device\", default=\"cpu\")\n    parser.add_argument(\"--model\", default=\"nori-6m\")\n    parser.add_argument(\"--features-only\", action=\"store_true\")\n    args = parser.parse_args()\n    rows = build_dataset(args.cache_dir, range(args.start_season, args.test_season+1))\n    rows.to_parquet(Path(args.cache_dir)/\"checkpoint_features.parquet\", index=False)\n    print(f\"Built {len(rows)} checkpoint rows, {len(feature_columns(rows))+len(CATEGORIES)} features\")\n    if not args.features_only:\n        pred = predict_weekly(rows, args.test_season, args.max_week, args.device, args.context_limit, model=args.model)\n        pred.to_parquet(Path(args.cache_dir)/\"predictions.parquet\", index=False)\n        metrics = prediction_metrics(pred)\n        print(metrics.to_string(index=False))\n        (Path(args.cache_dir)/\"metrics.json\").write_text(json.dumps(metrics.to_dict(\"records\"), indent=2))\n\n", "nfl_passing_yards_pipeline.py", "exec"), pipeline.__dict__)

In [ ]:
from pathlib import Path
import torch
CACHE_DIR = Path('nfl_passing_yards_cache')
OUTPUT_DIR = Path('nfl_passing_yards_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
START_SEASON = 2024
TEST_SEASON = 2025
MIN_WEEK, MAX_WEEK = 1, 1  # Set MAX_WEEK=18 for the full regular season.
CONTEXT_LIMIT = 128  # Fixed before evaluation; larger contexts cost more compute.
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
WEEKLY_UPDATE = True
RUN_MARKET_BACKTEST = False
rows = pipeline.build_dataset(CACHE_DIR, range(START_SEASON, TEST_SEASON+1))
rows.to_parquet(OUTPUT_DIR / 'checkpoint_features.parquet', index=False)
print(f'{len(rows)} rows; {len(pipeline.feature_columns(rows))+3} model columns')
display(rows[['season','week','game_id','actual_qb_name','checkpoint','yards_so_far','score_margin','qb_prior3_remaining_yards']].head())

## Generate fresh Nori predictions
Nori downloads its public weights on first use. Each checkpoint/week uses only completed earlier games as context. The context cap is explicit and affects predictions. fit() supplies examples; it does not train new model weights. Predict remaining yards, then add yards already thrown to every quantile.

Start with the default one-week CPU/GPU run to confirm the environment. A full-season run makes fresh predictions at both checkpoints every week and takes longer. The 2025 results are exploratory; freeze settings before a new evaluation season.

In [ ]:
predictions = pipeline.predict_weekly(rows, TEST_SEASON, max_week=MAX_WEEK,
    min_week=MIN_WEEK, device=DEVICE, context_limit=CONTEXT_LIMIT, weekly_update=WEEKLY_UPDATE)
predictions.to_parquet(OUTPUT_DIR / 'predictions.parquet', index=False)
metrics = pipeline.prediction_metrics(predictions)
metrics.to_csv(OUTPUT_DIR / 'forecast_metrics.csv', index=False)
display(metrics)
display(predictions[['game_id','actual_qb_name','checkpoint','official_passing_yards','predicted_mean','p10','p50','p90']].head())

## Visualize one forecast
The shaded interval contains the model’s central 80% of predicted final yardage. Coverage in the metric table measures how often that interval contained the actual result; nominal 80% is not a guarantee of calibration.

In [ ]:
import matplotlib.pyplot as plt
sample = predictions.iloc[0]
plt.figure(figsize=(8, 2.5))
plt.hlines(1, sample.p10, sample.p90, linewidth=12, color='#b8c5e6')
plt.scatter([sample.p50], [1], label='Predicted median', color='#1e2a78')
plt.scatter([sample.official_passing_yards], [1], marker='x', label='Actual final yards', color='#b45309')
plt.yticks([])
plt.xlabel('Final passing yards')
plt.title(f'{sample.actual_qb_name} — {sample.checkpoint}')
plt.legend()
plt.show()

## Optional: download historical Kalshi markets and replay quotes
Set RUN_MARKET_BACKTEST=True above to enable this step. It may make many public API requests and can resume from cache. It uses freshly generated predictions, not saved selections. Markets without a matching player/date or usable timestamped quote are reported as skipped. Q1 chooses highest qualifying edge; halftime is a fallback requiring same-line/side probability confirmation. This simplified market-selection policy differs from the original research configuration.

The report assumes one contract at the recorded ask, fees and 0/5/10¢ additional cost. It cannot establish available quantity or executable fills. A missing market is not a model failure, and a profitable simulation is not realized earnings.

In [ ]:
if RUN_MARKET_BACKTEST:
    markets = types.ModuleType("nfl_passing_yards_markets")
    sys.modules[markets.__name__] = markets
    exec(compile("\"\"\"Public, cached Kalshi quote-based replay. No live orders or assumed fills.\n\nThis deliberately uses public minute-candle closing quotes, not order-book depth.\nFreshly computed predictions need not reproduce any previously published result.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nfrom decimal import Decimal, ROUND_CEILING\nfrom pathlib import Path\nfrom urllib.parse import quote\n\nimport numpy as np\nimport pandas as pd\nimport requests\nfrom requests.adapters import HTTPAdapter\nfrom urllib3.util.retry import Retry\n\nBASE_URL = \"https://external-api.kalshi.com/trade-api/v2\"\n\n\nclass PublicKalshi:\n    \"\"\"Unauthenticated GET-only client; immutable request-keyed provenance cache.\"\"\"\n\n    def __init__(self, cache_dir):\n        self.cache_dir = Path(cache_dir) / \"kalshi\"\n        self.cache_dir.mkdir(parents=True, exist_ok=True)\n        self.session = requests.Session()\n        self.session.mount(\"https://\", HTTPAdapter(max_retries=Retry(\n            total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],\n            allowed_methods=[\"GET\"],\n        )))\n\n    def get(self, endpoint, **params):\n        url = f\"{BASE_URL}/{endpoint}\"\n        identity = json.dumps([url, params], sort_keys=True)\n        path = self.cache_dir / (hashlib.sha256(identity.encode()).hexdigest() + \".json\")\n        if path.exists():\n            record = json.loads(path.read_text())\n            if hashlib.sha256(json.dumps(record[\"payload\"], sort_keys=True).encode()).hexdigest() != record[\"payload_sha256\"]:\n                raise ValueError(f\"Corrupt cached payload: {path}\")\n            return record[\"payload\"]\n        response = self.session.get(url, params=params, timeout=60)\n        response.raise_for_status()\n        payload = response.json()\n        record = dict(url=response.url, retrieved_utc=pd.Timestamp.now(tz=\"UTC\").isoformat(),\n                      sha256=hashlib.sha256(response.content).hexdigest(),\n                      payload_sha256=hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest(), payload=payload)\n        temporary = path.with_suffix(\".tmp\")\n        temporary.write_text(json.dumps(record))\n        temporary.replace(path)\n        return payload\n\n    def markets(self):\n        \"\"\"Download all archived passing-yard markets, following every cursor.\"\"\"\n        result, cursor, seen = [], \"\", set()\n        while True:\n            params = {\"series_ticker\": \"KXNFLPASSYDS\", \"limit\": 1000}\n            if cursor:\n                params[\"cursor\"] = cursor\n            page = self.get(\"historical/markets\", **params)\n            result.extend(page[\"markets\"])\n            cursor = page.get(\"cursor\", \"\")\n            if not cursor:\n                break\n            if cursor in seen:\n                raise RuntimeError(\"Repeated historical-market cursor; download is incomplete\")\n            seen.add(cursor)\n        return list({m[\"ticker\"]: m for m in result}.values())\n\n    def quote_at(self, ticker, decision_utc, max_age_seconds=300):\n        \"\"\"Last completed minute candle at/before decision; never use future quotes.\"\"\"\n        decision = pd.Timestamp(decision_utc)\n        if pd.isna(decision) or decision.tzinfo is None:\n            raise ValueError(\"Decision timestamp must be known and timezone-aware\")\n        end = int(decision.timestamp())\n        payload = self.get(f\"historical/markets/{quote(ticker, safe='')}/candlesticks\",\n                           start_ts=end - max_age_seconds - 60, end_ts=end, period_interval=1)\n        candles = [c for c in payload[\"candlesticks\"]\n                   if 0 <= decision.timestamp() - c[\"end_period_ts\"] <= max_age_seconds]\n        if not candles:\n            return {}\n        candle = max(candles, key=lambda c: c[\"end_period_ts\"])\n        def close(field):\n            values = candle.get(field) or {}\n            if values.get(\"close_dollars\") is not None:\n                return float(values[\"close_dollars\"])\n            value = values.get(\"close\")\n            if value is None:\n                return None\n            # Historical fixed-point strings are dollars; legacy integer fields are cents.\n            return float(value) if isinstance(value, str) else float(value) / 100\n        return dict(quote_utc=pd.Timestamp(candle[\"end_period_ts\"], unit=\"s\", tz=\"UTC\").isoformat(),\n                    quote_age_seconds=decision.timestamp() - candle[\"end_period_ts\"],\n                    yes_bid=close(\"yes_bid\"), yes_ask=close(\"yes_ask\"))\n\n\ndef normalize_name(name):\n    return re.sub(r\"[^a-z0-9]\", \"\", str(name).lower())\n\n\ndef market_matches(market, row):\n    \"\"\"Require exact normalized player and original scheduled game date/team pair.\"\"\"\n    name = re.split(r\"\\s+records\\s+\\d+\\+|:\\s*\\d+\\+\", market.get(\"title\", \"\"),\n                    maxsplit=1, flags=re.I)[0]\n    if normalize_name(name) != normalize_name(row[\"actual_qb_name\"]):\n        return False\n    day = pd.Timestamp(row[\"game_date\"]).strftime(\"%y%b%d\").upper()\n    suffix = market.get(\"event_ticker\", \"\").removeprefix(\"KXNFLPASSYDS-\")\n    aliases = {\"LA\": \"LAR\", \"OAK\": \"LV\", \"SD\": \"LAC\"}\n    team = aliases.get(row[\"team\"], row[\"team\"])\n    opponent = aliases.get(row[\"opponent_team\"], row[\"opponent_team\"])\n    return suffix in (day + team + opponent, day + opponent + team)\n\n\ndef probability_over(levels, values, threshold):\n    levels, values = np.asarray(levels, float), np.asarray(values, float)\n    if (levels.ndim != 1 or values.shape != levels.shape or len(levels) < 2\n            or not np.isfinite(levels).all() or not np.isfinite(values).all()\n            or np.any(np.diff(levels) <= 0) or levels[0] <= 0 or levels[-1] >= 1):\n        raise ValueError(\"Invalid predictive quantiles\")\n    # Explicit piecewise-linear CDF approximation, monotone-repaired quantiles.\n    return float(1 - np.interp(threshold, np.maximum.accumulate(values), levels, left=0, right=1))\n\n\ndef taker_fee(price):\n    \"\"\"Modeled one-contract general taker fee, rounded upward to cents.\"\"\"\n    p = Decimal(str(price))\n    return float((Decimal(\"0.07\") * p * (1 - p)).quantize(Decimal(\"0.01\"), rounding=ROUND_CEILING))\n\n\ndef build_candidates(predictions, client, markets=None, max_age_seconds=300, max_spread=.10):\n    \"\"\"Return every matched side and rejected row, including missing data reasons.\n\n    Network errors stop the run (successful requests remain cached), rather than\n    silently treating failed downloads as absent liquidity.\n    \"\"\"\n    if predictions.empty:\n        raise ValueError(\"No predictions supplied\")\n    markets = client.markets() if markets is None else markets\n    rows = []\n    for row in predictions.to_dict(\"records\"):\n        base = {k: row[k] for k in [\"game_id\", \"week\", \"actual_qb_name\", \"checkpoint\", \"decision_utc\"]}\n        if pd.isna(row[\"decision_utc\"]):\n            rows.append({**base, \"reason\": \"missing_decision_timestamp\"})\n            continue\n        matches = [m for m in markets if market_matches(m, row) and m.get(\"floor_strike\") is not None]\n        if not matches:\n            rows.append({**base, \"reason\": \"no_matching_market\"})\n        for market in matches:\n            ticker, threshold = market[\"ticker\"], float(market[\"floor_strike\"])\n            if market.get(\"strike_type\") not in (\"greater\", \"structured\") or not re.search(r\"records \\d+\\+ passing yards\", market.get(\"title\", \"\"), re.I):\n                rows.append({**base, \"ticker\": ticker, \"reason\": \"unsupported_contract\"})\n                continue\n            p_yes = probability_over(row[\"quantile_levels\"], row[\"quantile_values\"], threshold)\n            snapshot = client.quote_at(ticker, row[\"decision_utc\"], max_age_seconds)\n            bid, ask = snapshot.get(\"yes_bid\"), snapshot.get(\"yes_ask\")\n            reason = \"eligible\"\n            if bid is None or ask is None:\n                reason = \"missing_quote\"\n            elif not 0 <= bid <= ask <= 1:\n                reason = \"invalid_quote\"\n            elif ask - bid > max_spread + 1e-9:\n                reason = \"wide_spread\"\n            settlement = market.get(\"settlement_value_dollars\")\n            if settlement is None:\n                settlement = {\"yes\": 1, \"no\": 0}.get(market.get(\"result\"))\n            if settlement is not None and not 0 <= float(settlement) <= 1:\n                raise ValueError(f\"Invalid settlement value for {ticker}\")\n            for side, probability, price in [(\"yes\", p_yes, ask), (\"no\", 1 - p_yes, None if bid is None else 1 - bid)]:\n                fee = taker_fee(price) if price is not None and 0 < price < 1 else None\n                rows.append({**base, **snapshot, \"ticker\": ticker, \"threshold\": threshold,\n                             \"line\": math.floor(threshold) + 1, \"side\": side,\n                             \"model_probability\": probability, \"entry_price\": price, \"fee\": fee,\n                             \"edge\": probability - price - fee if fee is not None else None,\n                             \"settlement_value\": None if settlement is None else (float(settlement) if side == \"yes\" else 1 - float(settlement)),\n                             \"reason\": \"nontradeable_price\" if reason == \"eligible\" and fee is None else reason})\n    return pd.DataFrame(rows)\n\n\ndef select_strategy(candidates, min_edge=.10):\n    \"\"\"Freeze Q1 first, halftime fallback; select highest edge deterministically.\n\n    Settlement never influences selection. Only one one-contract position per\n    QB/game. Halftime confirmation compares the identical ticker and side.\n    \"\"\"\n    audit = candidates.copy()\n    audit[\"selected\"] = False\n    for _, group in audit.groupby([\"game_id\", \"actual_qb_name\"], sort=False):\n        previous = {}\n        bought = False\n        for checkpoint in [\"q1\", \"halftime\"]:\n            choices = []\n            for index, row in group[group.checkpoint == checkpoint].iterrows():\n                key = (row.get(\"ticker\"), row.get(\"side\"))\n                if checkpoint == \"q1\" and pd.notna(row.get(\"model_probability\")):\n                    previous[key] = row[\"model_probability\"]\n                if row[\"reason\"] != \"eligible\":\n                    continue\n                reason = None\n                if bought:\n                    reason = \"earlier_entry\"\n                elif row[\"edge\"] < min_edge - 1e-12:\n                    reason = \"below_edge\"\n                elif checkpoint == \"halftime\" and key not in previous:\n                    reason = \"missing_q1_confirmation\"\n                elif checkpoint == \"halftime\" and row[\"model_probability\"] < previous[key]:\n                    reason = \"probability_not_confirmed\"\n                if reason:\n                    audit.at[index, \"reason\"] = reason\n                else:\n                    choices.append(index)\n            if choices:\n                winner = sorted(choices, key=lambda i: (-audit.at[i, \"edge\"], audit.at[i, \"ticker\"], audit.at[i, \"side\"]))[0]\n                for index in choices:\n                    audit.at[index, \"reason\"] = \"selected\" if index == winner else \"lower_priority\"\n                audit.at[winner, \"selected\"] = True\n                bought = True\n    selected = audit[audit.selected].copy()\n    selected[\"stake_contracts\"] = 1\n    if len(selected):\n        selected[\"capital\"] = selected.entry_price + selected.fee\n        selected[\"profit\"] = selected.settlement_value - selected.capital\n        for cents in (5, 10):\n            selected[f\"capital_plus_{cents}c\"] = selected.capital + cents / 100\n            selected[f\"profit_plus_{cents}c\"] = selected.profit - cents / 100\n    return audit, selected\n\n\ndef summarize(selected, allowances=(0, .05, .10)):\n    \"\"\"One-contract quote-based P&L, including each checkpoint and combined.\"\"\"\n    results = []\n    for checkpoint in [\"q1\", \"halftime\", \"combined\"]:\n        rows = selected if checkpoint == \"combined\" else selected[selected.checkpoint == checkpoint]\n        if len(rows) and rows.settlement_value.isna().any():\n            raise ValueError(\"Selected markets are unsettled: cannot report complete P&L\")\n        for allowance in allowances:\n            capital = float((rows.entry_price + rows.fee + allowance).sum()) if len(rows) else 0\n            payout = float(rows.settlement_value.sum()) if len(rows) else 0\n            results.append(dict(checkpoint=checkpoint, allowance=allowance, entries=len(rows),\n                                capital=capital, profit=payout-capital,\n                                return_on_deployed=(payout-capital)/capital if capital else None))\n    return pd.DataFrame(results)\n\n\ndef run_backtest(predictions, cache_dir, output_dir, **selection_options):\n    \"\"\"Download public data, write candidate reasons/entries/results, and return all.\"\"\"\n    candidates = build_candidates(predictions, PublicKalshi(cache_dir))\n    audit, selected = select_strategy(candidates, **selection_options)\n    report = summarize(selected)\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    audit.to_csv(output_dir / \"candidate_audit.csv\", index=False)\n    selected.to_csv(output_dir / \"selected_entries.csv\", index=False)\n    report.to_csv(output_dir / \"quote_based_results.csv\", index=False)\n    return audit, selected, report\n", "nfl_passing_yards_markets.py", "exec"), markets.__dict__)
    audit, entries, betting_metrics = markets.run_backtest(predictions, CACHE_DIR, OUTPUT_DIR, min_edge=.10)
    display(betting_metrics)
    display(audit.reason.value_counts())
else:
    print("Kalshi skipped. Fresh prediction and forecast evaluation are complete.")

In [ ]:
import json, hashlib
manifest = {'packages': {p: metadata.version(p) for p in required},
    'start_season':START_SEASON,'test_season':TEST_SEASON,'min_week':MIN_WEEK,'max_week':MAX_WEEK,
    'context_limit':CONTEXT_LIMIT,'weekly_update':WEEKLY_UPDATE,'model':'nori-6m','device':DEVICE,
    'sources':[json.loads(p.read_text()) for p in CACHE_DIR.glob('*.source.json')],
    'prediction_sha256':hashlib.sha256((OUTPUT_DIR/'predictions.parquet').read_bytes()).hexdigest()}
(OUTPUT_DIR/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Artifacts saved in', OUTPUT_DIR.resolve())